In [51]:
pip install torch numpy matplotlib

In [52]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.11.0+cu128
True
Tesla T4
cuda


In [53]:
import torch
import torch.nn as nn
from torch.nn import functional as F


In [54]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-05-27 04:34:30--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-05-27 04:34:30 (156 MB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [55]:
with open("input.txt","r", encoding="utf-8") as f:
  text=f.read()

print(text[::300])

F br
sehnremaohr  h eldtliiuUWuwe Caioepem Tr e bbotw
w:
a t
ghtiR doeooogc.e
ro:scnn.ild d aictan , hmeIaooIuieU li ,Urdatrek ahngwos Uu hlIw,ON Lt io BdnhtA bih h  tw Svgwadohdi
 hwr n ?t 
yer,.educ soOneFsirpctd eodsoiA efY Cy senohou
 ue Ooenntnee,euao iLw e'hvmessfaooRoeol uh Mortthtb rsttktmbldhm Irekt nn SAe eri:io:em .I,oN 
W ISut'h fuokhirloeFAoc Id
 fuhtgr dlke.drm : naBi c,urenu ,

nenoo
wo ait
suUux,hhaoc .. rwoI i',tkere
ge
ab tcn 
eeN magnh rtsOmtoeTa acaashengsi
tda'VehsoBodbIs'hlo
  sbnt ,psrBmott sain neGeI,use!bTS lor ueaAa'ecuTcC tu Ye tRNo  nTnWilel ao IhvoeeeIanlwnMsTO Om:rwnZai  nnsvhdlvU
oMua neniars  cFs h
rdthl' auf C
h 
olparpre hg sHe ana::r o:,Eu spU oaboIt f broTce si   tGaaC   h:
 ow ln ttpecta  gsoos ouRGe  yrTarhcula?GiR   :in wyeHEGl,os Te t.ds 
oaomsslw
 l Osnh N,heroerh  t,h  hae hrG
lh
e eer  tzswe fggr
isuw onerMn nometttteb  Semc c ogm p u
 h
cIvts ne- n n,
hntnu oeo
o,Qlulhioo sU iu a 
thle. nomTyD Zo:eee
ssgTp  dkfja,un R Ho fi to ekIouryei utn i

In [56]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [57]:
stoi={ch:i for i, ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])
print(encode("Hello"))
print(decode(encode(("Hello"))))

[20, 43, 50, 50, 53]
Hello


In [58]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [59]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [60]:
batch_size = 32
block_size = 64
n_embd = 64
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  return x.to(device), y.to(device)

x, y = get_batch("train")

print(x)
print(y)

tensor([[59, 50, 58,  ..., 57, 50, 43],
        [53, 53, 42,  ..., 59, 58,  1],
        [20, 47, 57,  ...,  1, 46, 47],
        ...,
        [43, 50, 50,  ..., 61, 39, 52],
        [58,  1, 53,  ..., 43, 56,  6],
        [ 5, 42,  1,  ..., 58, 46, 43]], device='cuda:0')
tensor([[50, 58,  1,  ..., 50, 43, 54],
        [53, 42,  1,  ..., 58,  1, 56],
        [47, 57,  1,  ..., 46, 47, 57],
        ...,
        [50, 50, 11,  ..., 39, 52, 58],
        [ 1, 53, 44,  ..., 56,  6,  0],
        [42,  1, 58,  ..., 46, 43,  1]], device='cuda:0')


In [61]:

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

In [62]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        out = self.proj(out)
        return out

In [63]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

In [64]:
class Block(nn.Module):
    def __init__(self, n_embd, num_heads):
        super().__init__()
        head_size = n_embd // num_heads
        self.sa = MultiHeadAttention(
            num_heads,
            head_size
        )
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [65]:
class TransformerLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )
        self.blocks = nn.Sequential(
            Block(n_embd, num_heads=4),
            Block(n_embd, num_heads=4),
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(
                logits,
                targets
            )
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(
                logits,
                dim=-1
            )
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )
        return idx

In [66]:
model = TransformerLanguageModel().to(device)

In [67]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)
for iter in range(max_iters):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if iter % eval_interval == 0:
        print(loss.item())

4.3560638427734375
2.533389091491699
2.4097800254821777
2.311493396759033
2.2412667274475098
2.1809375286102295
2.051480293273926
2.102764129638672
1.9473166465759277
1.9181082248687744


In [68]:
context = torch.zeros(
    (1,1),
    dtype=torch.long,
    device=device
)
generated_chars = model.generate(
    context,
    max_new_tokens=500
)[0].tolist()
print(decode(generated_chars))


RICHAS:
Till to priche sir: hell to to feathese!
Or not prizoall qon, nom,
Who dut of usomp black ques the to they dy hol 'fturmes?
Haw, strick as be louts'd is to untle.

HAEDWIM:
Fard I muke I now salloud thund a bame wee jut
Theek, in eyeth to wolink, last not
And dreri-plack with my forses hare,
Your have what nolt, live ap warraid' be hat Lod?

First shark,
That prupp be trange, me hom come fring,
That and to be pliciange, dehat in hee. I love pans thou but't of truesle
Which wo you stapore
